In [ ]:
import os
# Вимкнення зайвих логів TensorFlow тут щоб потім не вказувати це в кожному місці
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector

import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

In [ ]:
BASE_DIR = Path.cwd()
file_name = 'pokedex_b.csv'
DATA_PATH = BASE_DIR / 'data' / file_name
RANDOM_SEED = 42
TEST_SIZE = 0.2

try:
    df = pd.read_csv(DATA_PATH)
    print("File loaded successfully.")
except FileNotFoundError:
    print("File not found. Please check the file path and name.")


In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"### Shape: {df.shape[0]} rows, {df.shape[1]} columns"))
display(df.head())

display(Markdown("### Info"))
df.info()

Перевіряємо, наскільки збалансована кількість зразків кожного класу. В нашому випадку доля класу приблизно однаковою.

In [ ]:
df['is_legendary'].value_counts(normalize=True)

In [ ]:
df_cleaned = df.drop(columns=['name', 'pokedex_number'])
df_cleaned.head()

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(df_cleaned.corr(numeric_only=True), cmap='coolwarm', interpolation='nearest', vmin=-1, vmax=1)
plt.colorbar(label='Correlation Coefficient')
plt.title('Correlation Matrix')

# Додаємо назви колонок на осі
columns = df_cleaned.columns
plt.xticks(range(len(columns)), columns, rotation=45, ha='right')
plt.yticks(range(len(columns)), columns)

plt.tight_layout()
plt.show()

In [ ]:
df_cleaned = df_cleaned.join(pd.get_dummies(df_cleaned['type'], drop_first=True, prefix='type'))
df_cleaned.drop(columns=['type'], inplace=True)
df_cleaned.head()

In [ ]:
X = df_cleaned.drop(columns=['is_legendary']).values
y = df_cleaned['is_legendary'].values
y = to_categorical(y, num_classes=2)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
print(X_train.shape, y_train.shape)

In [ ]:
is_binary = np.all(np.isin(X_train, [0, 1]), axis=0)

cols_to_scale = ~is_binary

scaler = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), cols_to_scale),
    ],
    remainder='passthrough'
)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train = X_train.astype(float)
X_test = X_test.astype(float)

In [ ]:
model = Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu', kernel_regularizer= keras.regularizers.L2(0.001)),
    BatchNormalization(),
    Dropout(0.25),
    Dense(8, activation='relu', kernel_regularizer= keras.regularizers.L2(0.001)),
    BatchNormalization(),
    Dropout(0.25),
    Dense(2, activation='sigmoid', kernel_regularizer= keras.regularizers.L2(0.001)) 
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=64,          
    batch_size=16,      
    validation_split=0.1, 
    verbose=1,
    callbacks=[early]
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(loss, accuracy)

In [ ]:
plt.figure(figsize=(12, 5))

# Графік втрат (Loss)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Графік точності (Accuracy)
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import random

random_idx = random.randint(0, X_test.shape[0] - 1)
sample = X_test[random_idx].reshape(1, -1) 

prediction_probs = model.predict(sample, verbose=0)
predicted_class = np.argmax(prediction_probs) 
confidence = np.max(prediction_probs) * 100  

true_class = np.argmax(y_test[random_idx]) 

print(f"\n--- Результат передбачення для зразка #{random_idx} ---")
print(f"Істинний клас: {true_class}")
print(f"Передбачено клас {predicted_class} з достовірністю {confidence:.2f}%")

In [ ]:
model.save('islegendary.keras')